In [6]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random
from torch.optim import LBFGS, Adam
from tqdm import tqdm
import optuna

import copy
from model_components.models import DecoderOnlyPINNsformer, PINNs
from model_components.util import *
# Set seeds for reproducibility
seed = 0
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



def set_seed(seed=0):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
def h(x):
    return np.exp(- (x - np.pi)**2 / (2 * (np.pi/4)**2))

def u_ana(x, t):
    return h(x) * np.exp(5*t) / (h(x) * np.exp(5*t) + 1 - h(x))


print(device)


step_size = 1e-4
# Train PINNsformer
res, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 51, 51)
res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)

res, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 101, 101)
res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)

res = torch.tensor(res, dtype=torch.float32, requires_grad=True).to(device)
b_left = torch.tensor(b_left, dtype=torch.float32, requires_grad=True).to(device)
b_right = torch.tensor(b_right, dtype=torch.float32, requires_grad=True).to(device)
b_upper = torch.tensor(b_upper, dtype=torch.float32, requires_grad=True).to(device)
b_lower = torch.tensor(b_lower, dtype=torch.float32, requires_grad=True).to(device)

x_res, t_res = res[:,0:1], res[:,1:2]
x_left, t_left = b_left[:,0:1], b_left[:,1:2]
x_right, t_right = b_right[:,0:1], b_right[:,1:2]
x_upper, t_upper = b_upper[:,0:1], b_upper[:,1:2]
x_lower, t_lower = b_lower[:,0:1], b_lower[:,1:2]

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)
        


res_test = torch.tensor(res_test, dtype=torch.float32, requires_grad=True).to(device)
x_test, t_test = res_test[:,0:1], res_test[:,1:2]

res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)
u_true = u_ana(res_test[:,0], res_test[:,1]).reshape(101,101)

smallest_rl1 = 1e10  # Initialize a large value to track the smallest L1 error



set_seed(0)

d_hidden = 512
num_layer = 6


# Note: d_model (here 32) should be even for the Fourier features mapping.
model = PINNs(in_dim=2, hidden_dim=d_hidden, out_dim=1, num_layer=num_layer).to(device)

model.apply(init_weights)
optim = LBFGS(model.parameters(), line_search_fn='strong_wolfe')
# optim = Adam(model.parameters(), lr = 1e-4)



n_params = get_n_params(model)
kernel_size = 300

D1 = kernel_size
D2 = len(x_left)
D3 = len(x_lower)

def compute_ntk(J1, J2):
    Ker = torch.matmul(J1, torch.transpose(J2, 0, 1))
    return Ker

w1, w2, w3 = 1, 1, 1
model.load_state_dict(torch.load('saves/1drxn_spformer_trial_26.pth', map_location=device))





with torch.no_grad():
    pred = model(x_test, t_test)[:,0:1]
    pred = pred.cpu().detach().numpy()

pred = pred.reshape(101,101)


rl1 = np.sum(np.abs(u_true-pred)) / np.sum(np.abs(u_true))
rl2 = np.sqrt(np.sum((u_true-pred)**2) / np.sum(u_true**2))
n_params = get_n_params(model)

print(f"Relative L1 error: {rl1}, Relative L2 error: {rl2}, Number of parameters: {n_params}")

cuda
Relative L1 error: 0.01436103707648385, Relative L2 error: 0.02783300431056783, Number of parameters: 1052673
